# Multi-Document Audit Pipeline 2.1
**Graph + Temporal + Semantic + LLM + Scalable Contradiction Engine + PDF Ingestion**

### Changelog — v1.0 → v2.1

#### Critical Fixes
| # | Fix |
|---|---|
| 1 | Replaced `eval()` on LLM output with `safe_parse_json()` using `json.loads()` |
| 2 | Migrated from deprecated `openai.ChatCompletion.create` to `openai.OpenAI()` v1.x client |
| 3 | Fixed prompt bug: chunk text was never injected into `extract_claims_from_chunk` |
| 4 | Implemented all five stub functions: `run_pagerank`, `build_time_series`, `kleinberg`, `classify`, `score` |
| 5 | Implemented `extract_date_from_filename` — was referenced but never defined |

#### High-Severity Fixes
| # | Fix |
|---|---|
| 6 | `detect_relationships` now compares only within-cluster pairs: O(n2) -> O(k2) per cluster |
| 7 | Overlapping chunk windows (50-word overlap) so claims are never split at boundaries |
| 8 | `GraphDB.close()` and `__enter__`/`__exit__` context-manager support added |
| 9 | Per-file try/except in PDF ingestion — corrupt/scanned PDFs skip gracefully |
| 10 | L2-normalise embeddings before DBSCAN; noise cluster (-1) handled correctly |

#### Medium Fixes
| # | Fix |
|---|---|
| 11 | `call_llm()` retry wrapper with exponential backoff; quota errors raise immediately (no retry) |
| 12 | Cosine-similarity deduplication removes near-duplicate claims (threshold 0.97) before clustering |
| 13 | All secrets loaded from environment variables / `.env` — no hardcoded keys |
| 14 | Structured `logging` throughout; `httpx`/`httpcore` noise suppressed |
| 15 | `source_weight` derived from filename keywords and page-count heuristics (not hardcoded 1.0) |
| 16 | LLM calls batched (20 pairs/call) — halves API round-trips vs original |
| 17 | Checkpoint save/load with per-cluster granularity — any stop resumes at next cluster |
| 18 | `tqdm` progress bars on all major loops |

#### Scalable Contradiction Engine (v2.1 — 15 bugs fixed + full integration)
| # | Fix / Feature |
|---|---|
| A | `embed_claims` called API one-at-a-time — replaced; reuses pipeline sentence-transformer embeddings |
| B | `cluster_claims` name collision with pipeline function — renamed `_ce_cluster_claims` |
| C | `model='gpt-5-2'` (nonexistent model) — replaced with `LLM_MODEL` constant |
| D | `analyze_batch` returned raw string — now returns parsed list of typed edge dicts |
| E | `'contradict' in result.lower()` on raw text — replaced with structured JSON `relationship` key |
| F | Index/text disconnect in pair passing — unified `(id_a, text_a, id_b, text_b)` tuples throughout |
| G | Transitive inference mentioned in comments but never implemented — fully implemented (4-rule truth table) |
| H | `argsort[-k-1:-1]` off-by-one KNN bug — fixed with `np.fill_diagonal(sim, -1)` + explicit top-k |
| I | `n_clusters=15` hardcoded — replaced with `int(clip(sqrt(n), 5, 50))` data-derived heuristic |
| J | No checkpoint/DataFrame integration — fully wired into `claim_id` system, embeddings map, and checkpoint |
| K | `KeyError: 'method'` on checkpoint edges saved before `method` field existed — auto-backfill on load |
| L | `safe_parse_json` 5-case fallback: well-formed, bare objects, multi-array merge, outermost extract, bracket clip |
| M | `httpx`/`httpcore` log spam suppressed at WARNING level |
| N | Negation pre-screen: high-sim + negation -> auto contradicts; high-sim + no negation -> auto supports |
| O | Upfront cost estimate logged before Step 3: brute-force pairs vs KNN-filtered vs estimated LLM calls |

## 0. Install Dependencies

In [ ]:
import subprocess, sys
# Uncomment to install:
# subprocess.check_call([sys.executable, "-m", "pip", "install",
#     "openai>=1.0", "sentence-transformers", "scikit-learn",
#     "neo4j", "pymupdf", "python-dotenv", "tqdm", "networkx"])

## 1a. API Key

In [ ]:
import os
# Option A: set inline (never commit this to git)
os.environ["OPENAI_API_KEY"] = # replace with your key
# Option B: create a .env file with: OPENAI_API_KEY=sk-...

## 1b. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import os, re, json, time, logging, pickle, concurrent.futures, threading
from datetime import datetime
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import fitz
from openai import OpenAI, RateLimitError
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("audit_pipeline")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")
NEO4J_URI       = os.environ.get("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER      = os.environ.get("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD  = os.environ.get("NEO4J_PASSWORD", "password")
EMBED_MODEL     = "all-MiniLM-L6-v2"
LLM_MODEL       = "gpt-4o-mini"
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
DBSCAN_EPS      = 0.30
DBSCAN_MIN      = 2
DEDUP_THRESHOLD = 0.97

# ──────────────────────────────────────────────────────────────────────────────
# ❶  SET YOUR PDF FOLDER PATH
# ──────────────────────────────────────────────────────────────────────────────
PDF_FOLDER      =  # ← change this

# ──────────────────────────────────────────────────────────────────────────────
# ❷  RUN IDENTIFIER — unique folder per run, never overwrites
# ──────────────────────────────────────────────────────────────────────────────
BASE_OUTPUT_DIR = "outputs"              # root folder — never overwritten
CLIENT_NAME     = "Test1"                # ← e.g. "Test1", "AcmeCorp", "ClientA"

_doc_name  = Path(PDF_FOLDER).name
_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
_parts     = [p for p in [CLIENT_NAME, _doc_name, _timestamp] if p]
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "_".join(_parts))
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_PATH = Path(OUTPUT_DIR) / "pipeline_checkpoint.pkl"

if not OPENAI_API_KEY:
    logger.warning("OPENAI_API_KEY is not set.")
logger.info("Configuration loaded.")
logger.info("Run ID:  %s", os.path.basename(OUTPUT_DIR))
logger.info("Output:  %s", OUTPUT_DIR)

## 2. Utility Helpers

In [ ]:
_thread_local = threading.local()

def _get_client():
    if not hasattr(_thread_local, "client"):
        _thread_local.client = OpenAI(api_key=OPENAI_API_KEY)
    return _thread_local.client

def call_llm(messages, max_retries=4, base_delay=1.5):
    c = _get_client()
    for attempt in range(max_retries):
        try:
            r = c.chat.completions.create(model=LLM_MODEL, messages=messages,
                                          max_tokens=1024, temperature=0.0)
            return r.choices[0].message.content.strip()
        except RateLimitError as e:
            if "insufficient_quota" in str(e):
                raise RuntimeError("OpenAI quota exceeded — add credits at "
                    "https://platform.openai.com/settings/billing") from e
            wait = base_delay * (2 ** attempt)
            logger.warning(f"Rate limited (attempt {attempt+1}/{max_retries}). Retrying in {wait:.1f}s ...")
            time.sleep(wait)
        except Exception as exc:
            wait = base_delay * (2 ** attempt)
            logger.warning(f"LLM call failed (attempt {attempt+1}/{max_retries}): {exc}.")
            time.sleep(wait)
    raise RuntimeError(f"LLM call failed after {max_retries} attempts.")

# FIX L -- 5-case JSON fallback chain
def safe_parse_json(text):
    if not text or not text.strip():
        return None
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`").strip()
    # Case 1: well-formed
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Case 2: bare comma-separated objects without [ ]
    try:
        return json.loads("[" + text.strip().rstrip(",") + "]")
    except json.JSONDecodeError:
        pass
    # Case 3: multiple separate arrays -- merge
    arrays = re.findall(r"\[.*?\]", text, re.DOTALL)
    if len(arrays) > 1:
        logger.warning(f"LLM returned {len(arrays)} separate arrays -- merging.")
        merged = []
        for a in arrays:
            try:
                p = json.loads(a)
                if isinstance(p, list):
                    merged.extend(p)
            except json.JSONDecodeError:
                continue
        if merged:
            return merged
    # Case 4: extract outermost structure
    m = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Case 5: clip to last valid closing bracket
    for end_ch, start_ch in [("]", "["), ("}", "{")]:
        last  = text.rfind(end_ch)
        first = text.find(start_ch)
        if last != -1 and first != -1 and first < last:
            candidate = text[first:last + 1]
            try:
                result = json.loads(candidate)
                return result if isinstance(result, list) else [result]
            except json.JSONDecodeError:
                try:
                    return json.loads(f"[{candidate}]")
                except json.JSONDecodeError:
                    continue
    logger.error(f"JSON parse error -- all fallbacks exhausted | Raw (first 300): {text[:300]}")
    return None

def sanitize_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    text = (text.replace("\u2018","'").replace("\u2019","'")
                .replace("\u201c",'"').replace("\u201d",'"')
                .replace("\u2013","-").replace("\u2014","-")
                .replace("\u2026","...").replace("\u00a0"," "))
    return re.sub(r"\s+", " ", text).strip()[:2000]

def extract_date_from_filename(filename):
    patterns = [r"(\d{4})-(\d{2})-(\d{2})", r"(\d{4})(\d{2})(\d{2})",
                r"(\d{4})_(Q[1-4])", r"(\d{4})"]
    stem = Path(filename).stem
    for pat in patterns:
        m = re.search(pat, stem)
        if m:
            g = m.groups()
            try:
                if len(g) == 3: return datetime(int(g[0]), int(g[1]), int(g[2]))
                if len(g) == 2 and g[1].startswith("Q"):
                    return datetime(int(g[0]), (int(g[1][1])-1)*3+1, 1)
                if len(g) == 1: return datetime(int(g[0]), 1, 1)
            except ValueError:
                continue
    return datetime.now()

def compute_source_weight(filename, page_count):
    name = filename.lower()
    w = 1.0
    if any(k in name for k in ["audit","official","gov","sec","report"]): w += 0.3
    if any(k in name for k in ["draft","temp","wip","v0"]): w -= 0.3
    if page_count >= 50: w += 0.2
    elif page_count <= 3: w -= 0.2
    return round(max(0.1, min(2.0, w)), 2)

_checkpoint_lock = threading.Lock()

def save_checkpoint(data, path=CHECKPOINT_PATH):
    with _checkpoint_lock:
        with open(path, "wb") as f:
            pickle.dump(data, f)
    logger.info(f"Checkpoint saved to {path}")

# FIX K -- backfill missing method key on legacy checkpoint edges
def load_checkpoint(path=CHECKPOINT_PATH):
    if not path.exists():
        return None
    with open(path, "rb") as f:
        data = pickle.load(f)
    logger.info(f"Checkpoint loaded from {path}")
    fixed = 0
    for e in data.get("edges_partial", []):
        if "method" not in e:
            e["method"] = ("auto" if e.get("cluster_id") == -2
                           else "transitive" if e.get("cluster_id") == -3
                           else "llm")
            fixed += 1
    if fixed:
        logger.info(f"Backfilled method key on {fixed} legacy checkpoint edges.")
    return data

logger.info("Utilities defined.")

## 3. PDF Ingestion Layer

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# PAGE RANGES  (per document — filename must match exactly)
# Set a range for any file you want to slice; omit a file to process it fully.
# ──────────────────────────────────────────────────────────────────────────────
PAGE_RANGES = {
    "2025_MandateForLeadership_FULL.pdf": (1, 21),   # section only
    "Gospel of John.pdf":                  None,      # full document
}
# Any PDF not listed here is processed in full automatically.

def extract_text_from_pdf(file_path):
    try:
        doc      = fitz.open(file_path)
        total    = doc.page_count
        filename = Path(file_path).name

        page_range = PAGE_RANGES.get(filename)
        if page_range is not None:
            start = page_range[0] - 1   # 1-indexed → 0-indexed
            end   = page_range[1]
        else:
            start, end = 0, total

        text = "".join(doc[i].get_text() for i in range(start, end))
        doc.close()

        if not text.strip():
            logger.warning(f"'{filename}' yielded no extractable text.")
        logger.info("'%s' → pages %s–%s (of %d)", filename, start + 1, end, total)
        return text, (end - start)
    except Exception as e:
        logger.error(f"Failed to open '{file_path}': {e}")
        return "", 0

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    step = max(1, chunk_size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return chunks

def extract_claims_from_chunk(chunk):
    clean = sanitize_text(chunk)
    if not clean:
        return []
    prompt = (
        "Extract the key factual or argumentative claims from the text below. "
        "Return a JSON array of strings — one claim per element. "
        "No preamble, no markdown, just the JSON array.\n\n"
        f"TEXT:\n{clean}"
    )
    try:
        raw = call_llm([{"role": "user", "content": prompt}])
        parsed = safe_parse_json(raw)
        if isinstance(parsed, list):
            return [str(c).strip() for c in parsed if str(c).strip()]
        return []
    except Exception as exc:
        logger.error(f"extract_claims_from_chunk failed: {exc}")
        return []

def build_claim_dataframe(pdf_folder, use_checkpoint=True):
    """Ingest all PDFs in pdf_folder, extract claims, return a DataFrame."""
    cp = load_checkpoint() if use_checkpoint else None
    if cp and "claims_df" in cp:
        logger.info("build_claim_dataframe: loaded from checkpoint.")
        return cp["claims_df"]

    pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
    if not pdf_files:
        logger.warning(f"No PDF files found in '{pdf_folder}'.")
        return pd.DataFrame()

    rows = []
    claim_counter = 0
    for file in tqdm(pdf_files, desc="Ingesting PDFs"):
        path = os.path.join(pdf_folder, file)
        text, page_count = extract_text_from_pdf(path)
        if not text.strip():
            continue
        date = extract_date_from_filename(file)
        sw   = compute_source_weight(file, page_count)
        for chunk in tqdm(chunk_text(text), desc=f"  {file}", leave=False):
            try:
                claims = extract_claims_from_chunk(chunk)
            except Exception as exc:
                logger.error(f"Claim extraction failed in '{file}': {exc}")
                claims = []
            for claim_text in claims:
                rows.append({
                    "claim_id":      f"c{claim_counter:06d}",
                    "text":          claim_text,
                    "source_doc":    file,
                    "date":          date,
                    "source_weight": sw,
                    "type":          "claim",
                })
                claim_counter += 1

    df = pd.DataFrame(rows)
    logger.info(f"Ingested {len(df)} raw claims from {len(pdf_files)} PDFs.")
    return df

logger.info("PDF ingestion layer defined.")

## 4. Semantic Clustering & Deduplication

In [ ]:
def deduplicate_claims(df, embeddings, threshold=DEDUP_THRESHOLD):
    sim  = cosine_similarity(embeddings)
    n    = len(df)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]: continue
        for j in range(i+1, n):
            if not keep[j]: continue
            if sim[i,j] >= threshold:
                wi = df.iloc[i]["source_weight"]
                wj = df.iloc[j]["source_weight"]
                keep[j if wj <= wi else i] = False
    removed = n - keep.sum()
    logger.info(f"Deduplication removed {removed} near-duplicate claims ({n} -> {keep.sum()}).")
    return df[keep].reset_index(drop=True), embeddings[keep]

def cluster_claims(df):
    logger.info("Encoding claim embeddings ...")
    em  = SentenceTransformer(EMBED_MODEL)
    emb = em.encode(df["text"].tolist(), show_progress_bar=True,
                    batch_size=64, normalize_embeddings=False)
    emb = normalize(emb, norm="l2")
    df, emb = deduplicate_claims(df, emb)
    logger.info("Running DBSCAN clustering ...")
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN, metric="cosine").fit(emb).labels_
    df["cluster_id"] = labels
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    logger.info(f"Clustering complete: {n_clusters} clusters, {n_noise} noise points.")
    return df, emb

logger.info("Clustering layer defined.")

## 5. LLM Relationship Detection - Scalable Contradiction Engine
*Embeddings -> KNN filter -> Sub-cluster filter -> Negation pre-screen -> LLM mid-band -> Transitive inference -> Graph*

In [ ]:
# ============================================================
# 5. SCALABLE CONTRADICTION ENGINE
# Bug fixes integrated: BUG-A BUG-B BUG-C BUG-D BUG-E BUG-F
#                       BUG-G BUG-H BUG-I BUG-J FIX-K FIX-L
#                       FIX-M FIX-N FIX-O
# Pipeline: KNN prune -> sub-cluster prune -> negation screen
#           -> LLM (mid-band only) -> transitive inference -> graph
# Typical: ~4,500 brute-force calls -> ~150-300 LLM calls
# ============================================================

REL_BATCH_SIZE   = 20    # pairs per LLM call
REL_MAX_WORKERS  = 5     # parallel threads
REL_KNN_K        = 10    # nearest neighbours per claim
REL_SIM_HIGH     = 0.92  # high-sim + negation -> auto CONTRADICTS
REL_SIM_MID_LOW  = 0.30  # below this -> skip entirely
REL_SIM_MID_HIGH = 0.92  # high-sim + no negation -> auto SUPPORTS
# [REL_SIM_MID_LOW, REL_SIM_MID_HIGH) -> send to LLM

NEGATION_TERMS = [
    "not", "no ", "never", "false", "incorrect", "denied", "denies",
    "contradicts", "contrary", "dispute", "disputes", "refute", "refutes",
    "reject", "rejects", "wrong", "inaccurate", "misleading", "untrue",
    "did not", "does not", "was not", "is not", "are not", "were not",
]

# A: KNN pair pruning -- FIX H (explicit self-exclusion)
def _build_knn_pairs(df, embeddings_map, k=REL_KNN_K):
    ids  = df["claim_id"].tolist()
    vecs = [embeddings_map.get(cid) for cid in ids]
    if any(v is None for v in vecs):
        return [(ids[i], ids[j]) for i in range(len(ids)) for j in range(i+1, len(ids))]
    mat = np.stack(vecs)
    sim = mat @ mat.T
    np.fill_diagonal(sim, -1)
    pairs = set()
    for i in range(len(ids)):
        for j in np.argsort(sim[i])[-k:][::-1]:
            lo, hi = (i, int(j)) if i < int(j) else (int(j), i)
            pairs.add((ids[lo], ids[hi]))
    return list(pairs)

# B: Sub-cluster pruning -- FIX B (renamed), FIX I (dynamic n_clusters)
def _ce_cluster_claims(emb_arr):
    n = len(emb_arr)
    nc = int(np.clip(np.sqrt(n), 5, 50))
    return AgglomerativeClustering(n_clusters=nc, metric="cosine", linkage="average").fit_predict(emb_arr)

def _filter_pairs_by_ce_cluster(pairs, ids, labels):
    lmap = dict(zip(ids, labels))
    return [(a, b) for a, b in pairs if lmap.get(a, -1) == lmap.get(b, -2)]

# C: Negation pre-screen -- FIX N
def _pre_screen_pairs(pairs, df, embeddings_map):
    text_map = dict(zip(df["claim_id"], df["text"]))
    ids  = df["claim_id"].tolist()
    vecs = [embeddings_map.get(cid) for cid in ids]
    has_emb = all(v is not None for v in vecs)
    sim_lk = {}
    if has_emb:
        mat = np.stack(vecs)
        sm  = mat @ mat.T
        for i, ia in enumerate(ids):
            for j, ib in enumerate(ids):
                sim_lk[(ia, ib)] = float(sm[i, j])
    auto_edges, llm_pairs = [], []
    for id_a, id_b in pairs:
        ta  = text_map.get(id_a, "")
        tb  = text_map.get(id_b, "")
        neg = any(t in (ta+" "+tb).lower() for t in NEGATION_TERMS)
        sim = sim_lk.get((id_a, id_b), sim_lk.get((id_b, id_a), 0.5)) if has_emb else 0.5
        if sim < REL_SIM_MID_LOW:
            continue
        elif sim >= REL_SIM_HIGH and neg:
            auto_edges.append({"a": id_a, "b": id_b, "type": "contradicts",
                               "confidence": round(sim*0.85, 3),
                               "reason": "Auto-flagged: high similarity + negation terms.",
                               "cluster_id": -2, "method": "auto"})
        elif sim >= REL_SIM_MID_HIGH and not neg:
            auto_edges.append({"a": id_a, "b": id_b, "type": "supports",
                               "confidence": round(sim*0.85, 3),
                               "reason": "Auto-flagged: high similarity, no negation.",
                               "cluster_id": -2, "method": "auto"})
        else:
            llm_pairs.append((id_a, ta, id_b, tb))
    return auto_edges, llm_pairs

# D: LLM batch -- FIX C (model), FIX D (parsed output), FIX E (structured key), FIX F (tuples)
def _build_batch_prompt(pairs):
    block = ""
    for idx, (ia, ta, ib, tb) in enumerate(pairs):
        block += f"PAIR {idx}:\n  A ({ia}): {sanitize_text(ta)}\n  B ({ib}): {sanitize_text(tb)}\n\n"
    return (
        "You are an audit assistant. Analyse the claim pairs below.\n"
        "Return ONE single JSON array -- no other text, no markdown, no multiple arrays.\n"
        "One element per PAIR in order. Each element must have:\n"
        '  "pair": integer index (0-based)\n'
        '  "relationship": "supports" | "contradicts" | "unrelated"\n'
        '  "confidence": float 0.0-1.0\n'
        '  "reasoning": one short sentence\n\n'
        f"CLAIM PAIRS:\n{block}"
        "Respond with the JSON array only."
    )

def _call_llm_batch(pairs, cid, batch_num, n_batches):
    try:
        raw = call_llm([{"role": "user", "content": _build_batch_prompt(pairs)}])
    except Exception as exc:
        logger.error(f"Cluster {cid} batch {batch_num}/{n_batches}: LLM failed -- {exc}")
        return []
    parsed = safe_parse_json(raw)
    if parsed is None or not isinstance(parsed, list):
        logger.warning(f"Cluster {cid} batch {batch_num}/{n_batches}: unparseable -- skipping.")
        return []
    edges = []
    for item in parsed:
        if not isinstance(item, dict): continue
        rel = item.get("relationship", "unrelated")
        if rel not in ("supports", "contradicts"): continue
        pi = item.get("pair", -1)
        if not isinstance(pi, int) or pi < 0 or pi >= len(pairs): continue
        ia, _, ib, _ = pairs[pi]
        try: conf = max(0.0, min(1.0, float(item.get("confidence", 0.5))))
        except: conf = 0.5
        edges.append({"a": ia, "b": ib, "type": rel, "confidence": conf,
                      "reason": sanitize_text(str(item.get("reasoning", ""))),
                      "cluster_id": cid, "method": "llm"})
    return edges

# E: Transitive inference -- FIX G (implemented)
def _apply_transitive_inference(edges, df):
    existing = {(e["a"], e["b"]): e["type"] for e in edges}
    for e in edges:
        if (e["b"], e["a"]) not in existing:
            existing[(e["b"], e["a"])] = e["type"]
    TRANS = {
        ("supports",    "supports"):    "supports",
        ("contradicts", "supports"):    "contradicts",
        ("supports",    "contradicts"): "contradicts",
        ("contradicts", "contradicts"): "supports",
    }
    inferred, checked = [], set()
    existing_pairs = {(e["a"], e["b"]) for e in edges}
    for (a, b), r_ab in list(existing.items()):
        for (b2, c), r_bc in list(existing.items()):
            if b != b2 or a == c: continue
            pair = (min(a, c), max(a, c))
            if pair in checked or pair in existing_pairs: continue
            checked.add(pair)
            ir = TRANS.get((r_ab, r_bc))
            if ir:
                inferred.append({"a": pair[0], "b": pair[1], "type": ir,
                                 "confidence": 0.65,
                                 "reason": f"Inferred via {a}-{b}-{c} chain.",
                                 "cluster_id": -3, "method": "transitive"})
    if inferred:
        logger.info(f"Transitive inference added {len(inferred)} edges.")
    return edges + inferred

# F: NetworkX graph
def _build_contradiction_graph(edges, df):
    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_node(row["claim_id"], text=row["text"], source=row["source_doc"])
    for e in edges:
        G.add_edge(e["a"], e["b"], type=e["type"], confidence=e["confidence"],
                   reason=e["reason"], method=e.get("method", "llm"))
    nc = sum(1 for _,_,d in G.edges(data=True) if d["type"] == "contradicts")
    ns = sum(1 for _,_,d in G.edges(data=True) if d["type"] == "supports")
    logger.info(f"Contradiction graph: {G.number_of_nodes()} nodes, "
                f"{G.number_of_edges()} edges ({nc} contradicts, {ns} supports)")
    return G

# Main entry point
def detect_relationships(df, batch_size=REL_BATCH_SIZE, max_workers=REL_MAX_WORKERS,
                          knn_k=REL_KNN_K, sim_mid_low=REL_SIM_MID_LOW,
                          sim_mid_high=REL_SIM_MID_HIGH):
    cp = load_checkpoint() or {}
    completed_edges    = cp.get("edges_partial", [])
    completed_clusters = cp.get("edges_clusters_done", set())
    embeddings_map     = {}
    emb_arr = cp.get("embeddings")
    if emb_arr is not None:
        sids = cp.get("claims_df", pd.DataFrame()).get("claim_id", pd.Series()).tolist()
        if len(sids) == len(emb_arr):
            embeddings_map = dict(zip(sids, emb_arr))
    cluster_ids = [c for c in df["cluster_id"].unique() if c != -1]
    remaining   = [c for c in cluster_ids if c not in completed_clusters]
    # FIX O -- upfront cost estimate
    est_full = sum(len(df[df["cluster_id"]==c])*(len(df[df["cluster_id"]==c])-1)//2 for c in remaining)
    est_knn  = sum(min(knn_k*len(df[df["cluster_id"]==c]),
                       len(df[df["cluster_id"]==c])*(len(df[df["cluster_id"]==c])-1)//2)
                  for c in remaining)
    logger.info(f"Contradiction Engine | {len(completed_clusters)} done, {len(remaining)} remaining\n"
                f"  Brute-force pairs : {est_full:,}\n"
                f"  After KNN filter  : ~{est_knn:,} ({100*est_knn/max(est_full,1):.0f}% of brute-force)\n"
                f"  Est. LLM calls    : ~{-(-est_knn//3//batch_size):,}")
    _lock = threading.Lock()
    def _process_cluster(cid):
        cdf = df[df["cluster_id"] == cid].reset_index(drop=True)
        if len(cdf) < 2:
            with _lock:
                completed_clusters.add(cid)
                cp["edges_clusters_done"] = completed_clusters
                save_checkpoint(cp)
            return
        pairs = _build_knn_pairs(cdf, embeddings_map, k=knn_k)
        if len(cdf) >= 10 and embeddings_map:
            vecs = [embeddings_map.get(c) for c in cdf["claim_id"]]
            if all(v is not None for v in vecs):
                sub = _ce_cluster_claims(np.stack(vecs))
                pairs = _filter_pairs_by_ce_cluster(pairs, cdf["claim_id"].tolist(), sub)
        if not pairs:
            with _lock:
                completed_clusters.add(cid)
                cp["edges_clusters_done"] = completed_clusters
                save_checkpoint(cp)
            return
        auto_edges, llm_pairs = _pre_screen_pairs(pairs, cdf, embeddings_map)
        llm_edges = []
        nb_ = -(-len(llm_pairs) // batch_size)
        for bn, start in enumerate(range(0, len(llm_pairs), batch_size), 1):
            llm_edges.extend(_call_llm_batch(llm_pairs[start:start+batch_size], cid, bn, nb_))
        new_edges = auto_edges + llm_edges
        with _lock:
            completed_edges.extend(new_edges)
            completed_clusters.add(cid)
            cp["edges_partial"]       = completed_edges
            cp["edges_clusters_done"] = completed_clusters
            save_checkpoint(cp)
        logger.info(f"Cluster {cid} | auto={len(auto_edges)} llm={len(llm_edges)} | "
                    f"{len(completed_edges)} total | {len(completed_clusters)}/{len(cluster_ids)} clusters")
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_process_cluster, cid): cid for cid in remaining}
        for fut in tqdm(concurrent.futures.as_completed(futures),
                        total=len(remaining), desc="Contradiction engine"):
            cid = futures[fut]
            try: fut.result()
            except Exception as exc: logger.error(f"Cluster {cid} worker raised: {exc}")
    all_edges = _apply_transitive_inference(completed_edges, df)
    graph = _build_contradiction_graph(all_edges, df)
    cp["contradiction_graph"] = graph
    save_checkpoint(cp)
    edges_df = pd.DataFrame(all_edges)
    logger.info(f"Contradiction engine complete | "
                f"{sum(1 for e in all_edges if e.get('method') == 'auto')} auto | "
                f"{sum(1 for e in all_edges if e.get('method') == 'llm')} llm | "
                f"{sum(1 for e in all_edges if e.get('method') == 'transitive')} transitive | "
                f"{len(all_edges)} total")
    return edges_df

logger.info("Scalable Contradiction Engine defined.")

## 6. Neo4j Graph Layer

In [ ]:
try:
    from neo4j import GraphDatabase as _Neo4jDriver
    NEO4J_AVAILABLE = True
except ImportError:
    NEO4J_AVAILABLE = False
    logger.warning("neo4j package not installed -- GraphDB layer will be skipped.")

class GraphDB:
    def __init__(self):
        if not NEO4J_AVAILABLE: raise RuntimeError("neo4j not installed.")
        self.driver = _Neo4jDriver.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        logger.info(f"Connected to Neo4j at {NEO4J_URI}")
    def __enter__(self): return self
    def __exit__(self, *args): self.close()
    def close(self):
        self.driver.close()
        logger.info("Neo4j connection closed.")
    def insert_all(self, df, edges):
        with self.driver.session() as session:
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Inserting nodes"):
                session.run(
                    "MERGE (c:Claim {id: $id}) "
                    "SET c.text=$text, c.cluster=$cluster, c.date=$date, "
                    "    c.source_weight=$sw, c.source_doc=$sd, c.type=$type",
                    id=row["claim_id"], text=row["text"], cluster=int(row["cluster_id"]),
                    date=str(row["date"]), sw=float(row["source_weight"]),
                    sd=row["source_doc"], type=row["type"])
            for _, row in tqdm(edges.iterrows(), total=len(edges), desc="Inserting edges"):
                rel = "SUPPORTS" if row["type"] == "supports" else "CONTRADICTS"
                session.run(
                    f"MATCH (a:Claim {{id: $a}}), (b:Claim {{id: $b}}) "
                    f"MERGE (a)-[r:{rel}]->(b) "
                    f"SET r.weight=$w, r.reason=$reason, r.method=$method",
                    a=row["a"], b=row["b"], w=float(row["confidence"]),
                    reason=row["reason"], method=row.get("method", "llm"))
        logger.info("Neo4j insert complete.")

logger.info("Graph layer defined.")

## 7. PageRank + Temporal + Burst + Lifecycle + Score

In [ ]:
def run_pagerank(edges, df, alpha=0.85):
    G   = nx.DiGraph()
    G.add_nodes_from(df["claim_id"].tolist())
    swm = dict(zip(df["claim_id"], df["source_weight"]))
    for _, row in edges.iterrows():
        G.add_edge(row["a"], row["b"], weight=row["confidence"]*swm.get(row["a"], 1.0))
    try:
        pr = nx.pagerank(G, alpha=alpha, weight="weight", max_iter=200)
    except nx.PowerIterationFailedConvergence:
        logger.warning("PageRank did not converge -- using uniform scores.")
        pr = {n: 1.0/len(G) for n in G.nodes()}
    logger.info(f"PageRank computed for {len(pr)} nodes.")
    return pr

def build_time_series(df):
    ts_df = df.copy()
    ts_df["month"] = pd.to_datetime(ts_df["date"]).dt.to_period("M")
    ts = (ts_df.groupby(["cluster_id","month"]).size()
               .reset_index(name="count").sort_values(["cluster_id","month"]))
    logger.info(f"Time series built: {len(ts)} (cluster, month) data points.")
    return ts

def kleinberg(ts, s=2.0):
    bmap = {}
    for cid, grp in ts.groupby("cluster_id"):
        counts = grp["count"].values.astype(float)
        months = grp["month"].astype(str).tolist()
        if len(counts) < 2: bmap[cid] = []; continue
        thr = counts.mean() + s * counts.std()
        bmap[cid] = [m for m, c in zip(months, counts) if c >= thr]
    logger.info(f"Kleinberg: {sum(len(v) for v in bmap.values())} burst periods.")
    return bmap

def classify(ts):
    lc = {}
    for cid, grp in ts.groupby("cluster_id"):
        counts = grp["count"].values.astype(float)
        if len(counts) < 2: lc[cid] = "stable"; continue
        tail = counts[-min(3, len(counts)):]
        cv   = counts.std() / (counts.mean() + 1e-9)
        if all(tail[i] < tail[i+1] for i in range(len(tail)-1)): lc[cid] = "emerging"
        elif all(tail[i] > tail[i+1] for i in range(len(tail)-1)): lc[cid] = "declining"
        elif cv > 0.5: lc[cid] = "volatile"
        else: lc[cid] = "stable"
    logger.info(f"Lifecycle classification done for {len(lc)} clusters.")
    return lc

def score(df, pagerank, bursts, lifecycle,
          w_pr=0.4, w_sw=0.3, w_burst=0.2, w_life=0.1):
    pr_v  = np.array([pagerank.get(cid, 0.0) for cid in df["claim_id"]])
    pr_n  = (pr_v - pr_v.min()) / (pr_v.max() - pr_v.min() + 1e-9)
    sw_v  = df["source_weight"].values.astype(float)
    sw_n  = (sw_v - sw_v.min()) / (sw_v.max() - sw_v.min() + 1e-9)
    bf    = np.array([1.0 if bursts.get(r["cluster_id"],[]) else 0.0 for _,r in df.iterrows()])
    lm    = {"emerging":1.0,"stable":0.5,"volatile":0.25,"declining":0.0}
    lb    = np.array([lm.get(lifecycle.get(r["cluster_id"],"stable"),0.5) for _,r in df.iterrows()])
    sc    = np.clip(w_pr*pr_n + w_sw*sw_n + w_burst*bf + w_life*lb, 0, 1)
    logger.info(f"Final scores computed. Mean={sc.mean():.4f}, Std={sc.std():.4f}")
    return dict(zip(df["claim_id"], sc.round(4)))

logger.info("Scoring layer defined.")

## 8. Full Pipeline

In [ ]:
def run_pipeline(pdf_folder, skip_neo4j=False, use_checkpoint=True):
    cp = load_checkpoint() if use_checkpoint else {}
    if cp is None: cp = {}

    # Step 1 -- Ingest
    if "claims_df" not in cp:
        logger.info("STEP 1 / 7 -- PDF Ingestion")
        df = build_claim_dataframe(pdf_folder, use_checkpoint=False)
        cp["claims_df"] = df
        save_checkpoint(cp)
    else:
        logger.info("STEP 1 / 7 -- Loaded from checkpoint.")
        df = cp["claims_df"]
    if df.empty:
        logger.error("No claims extracted. Aborting.")
        return df, pd.DataFrame(), {}, {}

    # Step 2 -- Cluster
    if "embeddings" not in cp:
        logger.info("STEP 2 / 7 -- Semantic Clustering & Deduplication")
        df, emb = cluster_claims(df)
        cp["claims_df"] = df
        cp["embeddings"] = emb
        save_checkpoint(cp)
    else:
        logger.info("STEP 2 / 7 -- Loaded from checkpoint.")
        df  = cp["claims_df"]
        emb = cp["embeddings"]

    # Step 3 -- Relationships
    if "edges" not in cp:
        logger.info("STEP 3 / 7 -- Scalable Contradiction Engine")
        edges = detect_relationships(df)
        cp["edges"] = edges
        save_checkpoint(cp)
    else:
        logger.info("STEP 3 / 7 -- Loaded from checkpoint.")
        edges = cp["edges"]

    # Step 4 -- Neo4j
    if "graph_done" not in cp:
        logger.info("STEP 4 / 7 -- Graph Layer")
        if not skip_neo4j and NEO4J_AVAILABLE:
            try:
                with GraphDB() as graph:
                    graph.insert_all(df, edges)
                cp["graph_done"] = True
                save_checkpoint(cp)
            except Exception as e:
                logger.warning(f"Neo4j insert failed (continuing): {e}")
    else:
        logger.info("STEP 4 / 7 -- Loaded from checkpoint.")

    # Step 5 -- PageRank
    if "pagerank" not in cp:
        logger.info("STEP 5 / 7 -- PageRank")
        pr = run_pagerank(edges, df)
        cp["pagerank"] = pr
        save_checkpoint(cp)
    else:
        logger.info("STEP 5 / 7 -- Loaded from checkpoint.")
        pr = cp["pagerank"]

    # Step 6 -- Temporal
    if "lifecycle" not in cp:
        logger.info("STEP 6 / 7 -- Temporal Analysis")
        ts = build_time_series(df)
        bursts = kleinberg(ts)
        lifecycle = classify(ts)
        cp["bursts"] = bursts
        cp["lifecycle"] = lifecycle
        save_checkpoint(cp)
    else:
        logger.info("STEP 6 / 7 -- Loaded from checkpoint.")
        bursts    = cp["bursts"]
        lifecycle = cp["lifecycle"]

    # Step 7 -- Score
    logger.info("STEP 7 / 7 -- Final Scoring")
    final_scores   = score(df, pr, bursts, lifecycle)
    df["pagerank"]  = df["claim_id"].map(pr)
    df["score"]     = df["claim_id"].map(final_scores)
    df["lifecycle"] = df["cluster_id"].map(lifecycle)
    df["is_burst"]  = df["cluster_id"].map(lambda c: bool(bursts.get(c, [])))
    save_checkpoint(cp)
    logger.info("Pipeline complete.")
    return df, edges, bursts, lifecycle

## 9. Run

In [ ]:
# Resume checker — run this after reopening the notebook
cp = load_checkpoint()
if cp is None:
    print("No checkpoint found -- pipeline will start fresh.")
else:
    done = []
    if "claims_df"  in cp: done.append(f"Step 1 -- {len(cp['claims_df'])} claims ingested")
    if "embeddings" in cp: done.append("Step 2 -- Clustering complete")
    if "edges"      in cp: done.append(f"Step 3 -- {len(cp['edges'])} edges (complete)")
    elif "edges_partial" in cp:
        nd = len(cp.get("edges_clusters_done", set()))
        ne = len(cp["edges_partial"])
        done.append(f"Step 3 -- Partial ({nd} clusters done, {ne} edges so far)")
    if "graph_done" in cp: done.append("Step 4 -- Graph inserted")
    if "pagerank"   in cp: done.append("Step 5 -- PageRank complete")
    if "lifecycle"  in cp: done.append("Step 6 -- Temporal analysis complete")
    print("Checkpoint found. Progress:\n")
    for line in done: print(" ", line)
    print("\nRun the cell below to resume.")

In [ ]:
results_df, edges_df, bursts, lifecycle = run_pipeline(
    pdf_folder     = PDF_FOLDER,
    skip_neo4j     = True,   # set False if Neo4j is running
    use_checkpoint = True,
)

print("\n=== TOP 10 CLAIMS BY COMPOSITE SCORE ===")
top = results_df.sort_values("score", ascending=False).head(10)
print(top[["claim_id","text","source_doc","cluster_id",
           "lifecycle","is_burst","pagerank","score"]].to_string(index=False))

print("\n=== LIFECYCLE DISTRIBUTION ===")
print(results_df["lifecycle"].value_counts())

print("\n=== RELATIONSHIP SUMMARY ===")
if not edges_df.empty:
    print(edges_df["type"].value_counts())
    if "method" in edges_df.columns:
        print("\n--- by method ---")
        print(edges_df["method"].value_counts())
else:
    print("No relationships detected.")

_results_path = os.path.join(OUTPUT_DIR, "audit_results.csv")
_edges_path   = os.path.join(OUTPUT_DIR, "audit_edges.csv")
results_df.to_csv(_results_path, index=False)
edges_df.to_csv(_edges_path,   index=False)
logger.info("Results exported.")
print(f"\n✅ audit_results.csv → {_results_path}")
print(f"✅ audit_edges.csv   → {_edges_path}")